In [ ]:
# 1. 구글드라이브 연동 및 깃허브 클론/풀 (동일한 환경 세팅)
import os
import sys
import shutil
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt




In [ ]:

# 2. 필요한 파일들 로드 (vocab.json 및 구글 드라이브에서 model.pt 가져오기)
import shutil, os

os.makedirs("models", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/models/vocab.json", "models/vocab.json")

# 구글 드라이브에 저장되어 있는 학습 완료된 model.pt 파일을 복사해옵니다.
drive_model_path = "/content/drive/MyDrive/korean-chatbot/models/model.pt"
local_model_path = "models/model.pt"

if os.path.exists(drive_model_path):
    shutil.copy(drive_model_path, local_model_path)
    print("✅ 구글 드라이브로부터 model.pt 로드 완료!")
else:
    print(f"❌ 구글 드라이브 경로에 모델 파일이 없습니다: {drive_model_path}")

In [ ]:


# 3. GPU 장치 정의 및 A100 가속 활성화
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

if torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("🚀 A100 GPU 감지: TF32 매트릭스 연산 가속을 켭니다.")

In [ ]:

# 4. 모델 및 토크나이저 로드 및 설정
# ※ 주의: 이 셀을 실행하기 전에 대포적인 깃허브 저장소 내에 model과 tok 클래스가 정의되어 있거나,
#         코드 상단에 model과 tok의 선언(초기화)이 완료되어 있어야 합니다.

if os.path.exists(local_model_path):
    # 가중치 파일 로드 및 모델을 GPU/CPU 장치로 이동
    model.load_state_dict(torch.load(local_model_path, map_location=device))
    model = model.to(device)
    model.eval()  # 추론 모드 전환 (Dropout, BatchNorm 비활성화)
    print("✨ 가중치 모델 적용 완료! 추론 준비가 되었습니다.")
else:
    print("❌ 로컬에 model.pt 파일이 존재하지 않아 가중치를 로드하지 못했습니다.")

In [ ]:

# 5. 테스트 입력 및 문장 생성 (Inference 실행)
test_inputs = [
    # 인사
    "안녕하세요",
    "반갑습니다",
    "좋은 아침이에요",
    
    # 한국 역사
    "조선은",
    "한국의 역사는",
    "고려시대에는",
    "삼국시대란",
    
    # 지식
    "인공지능이란",
    "머신러닝은",
    "딥러닝의 원리는",
    "트랜스포머 모델은",
    
    # 나무위키 스타일
    "대한민국은",
    "서울은",
    "한국어란",
    "김치는",
    
    # 문장 완성
    "오늘 날씨가",
    "밥을 먹으러",
    "학교에서",
    "회사에서",
]

print("\n🤖 문장 생성을 시작합니다...\n")
print("=" * 50)

# 가속화된 추론 환경 구축을 위한 컨텍스트 매니저 설정
with torch.no_grad():
    # 학습 때와 동일하게 bfloat16 AMP 가속을 활용해 추론 속도를 높입니다.
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32):
        for text in test_inputs:
            result = model.generate(text, tok)
            print(f"입력: {text}")
            print(f"출력: {result}")
            print("-" * 50)

print("\n🎉 모든 추론이 완료되었습니다!")

In [ ]:
while True:
    prompt = input("입력 ('q'로 종료): ")
    if prompt == 'q':
        break
    result = engine.generate(prompt, max_new_tokens=100, temperature=0.8)
    print(f"출력: {result}\n")